# 4. Statistiek van de hoogwaterstand, overige stations

Python-versie van het R-script **`statistics-wl10others-impr.Rmd`** (KNMI, februari 2026).

Voor OS11, Den Oever, West-Terschelling, Lauwersoog en Huibertgat zijn de metingen te kort of niet geschikt
voor een eigen analyse. De herhalingstijdwaarden worden afgeleid uit SEAS5-DCSM5 en uit referentiestations:

1. fit GW-staarten aan de SEAS5-DCSM5-data van het station zelf;
2. corrigeer de omnidirectionele herhalingstijdwaarden met de verhouding metingen / SEAS5-DCSM5 van een
   *omni-referentiestation*, plus een vaste verschuiving; neem de richtingsafhankelijkheid over van een
   *richtingsreferentiestation*; neem de onzekerheid over uit de bootstrap van het omni-referentiestation.

| station | omni-referentie | richtingsreferentie | verschuiving [m] |
|---|---|---|---|
| os11 | hoekvanh | vlissing | 0,0 |
| denoever | denhelde | denhelde | 0,1 |
| westters | harlinge | harlinge | 0,05 |
| lauwerso | harlinge | harlinge | 0,2 |
| huibertg | harlinge | harlinge | 0,0 |

**Gebruik:** voer eerst `3_statistics_wl.ipynb` uit (dat maakt de resultaten van de referentiestations),
controleer de instellingen hieronder en voer daarna alle cellen uit.

Toevalsgetallen (`set.seed(180225)`) worden alleen in de laatste stap gebruikt, voor de metingen, die R
alleen voor figuren gebruikt. Elk station krijgt daarbij precies de toevalsgetallen van de R-run, ook als je maar
één station of de stations in een andere volgorde doorrekent.

Volgend notebook: `5_statistics_wl_texeln.ipynb`.

## Instellingen
In `stations` mag je elke selectie van stations zetten, in elke volgorde. `r_stations` is de stationslijst van de
R-run: daaruit volgen de toevalsgetallen, dus die pas je normaal niet aan.
De standaardwaarden van de mappen en van `r_stations` staan in `kuststatistiek/settings.py`.

In [ ]:
from kuststatistiek import settings

input_path = settings.DATA_PATH  # map met de aangeleverde invoerdata
output_path = (
    settings.OUTPUT_PATH
)  # map voor de resultaten (hier worden ook de resultaten van notebook 3 gelezen)
r_stations = settings.OTHER_STATIONS  # stationslijst van de R-run, in de volgorde van R

# stations die je wilt doorrekenen (bijvoorbeeld alleen ["os11"])
stations = ["os11", "denoever", "westters", "lauwerso", "huibertg"]

## Voorbereiding

In [ ]:
from kuststatistiek.settings import (
    Paths,
    Bins,
    wl_settings,
    OTHER_STATIONS,
    OTHER_OMNIREF,
    OTHER_DIRREF,
    OTHER_SHIFTS,
    OTHER_SEED,
    WL_MEASFILE,
)
from kuststatistiek.rstream import others_random_streams
from kuststatistiek.io.read_wind import read_seas5_multi
from kuststatistiek.io.read_wl import read_station_wlmeas
from kuststatistiek.analysis import fit_seas5_tails, combine_other_station
from kuststatistiek.rutils import save_pkl, load_pkl
from kuststatistiek.plotting import plot_returnlevels

for station in stations:
    if station not in r_stations:
        raise ValueError(f"station {station} staat niet in r_stations")

paths = Paths(input_path=input_path, output_path=output_path)
WL = wl_settings()  # constanten van het R-script
bins = Bins()  # richtingssectoren

# referentiestations en verschuiving per station (tabel hierboven)
omniref = dict(zip(OTHER_STATIONS, OTHER_OMNIREF))
dirref = dict(zip(OTHER_STATIONS, OTHER_DIRREF))
shift = dict(zip(OTHER_STATIONS, OTHER_SHIFTS))

## Stap 1 – GW-staarten uit SEAS5-DCSM5 (R-chunk `dir-es-seas5dcms5`)
Uitvoer: `<station>_stats_multi.pkl`

In [ ]:
statdata = {}
for station in stations:
    wl_dcsm = read_seas5_multi(station, paths.input_path)
    statdata[station] = fit_seas5_tails(
        wl_dcsm["wl"], wl_dcsm["dwind"], WL, bins, caseId=station
    )
    save_pkl(paths.out(f"{station}_stats_multi.pkl"), **statdata[station])
    print(f"{station}: GW-fit SEAS5-DCSM5 klaar")

## Stap 2 – Combineren met de referentiestations (R-chunk `combine`)
Gebruikt `<ref>_stats_multi.pkl` en `<ref>_stats_multi_meas2.pkl` van de referentiestations (notebook 3).

Uitvoer: `<station>_stats_multi_meas3.pkl` (gebruikt voor de tabellen in `6_production_wl.ipynb`)

In [ ]:
meas3 = {}
for station in stations:
    o, d = omniref[station], dirref[station]
    meas3[station] = combine_other_station(
        own_stats_multi=statdata[station],
        omniref_stats_multi=load_pkl(paths.out(f"{o}_stats_multi.pkl")),
        omniref_meas2=load_pkl(paths.out(f"{o}_stats_multi_meas2.pkl")),
        dirref_stats_multi=load_pkl(paths.out(f"{d}_stats_multi.pkl")),
        dirref_meas2=load_pkl(paths.out(f"{d}_stats_multi_meas2.pkl")),
        shift=shift[station],
        var=WL,
        bins=bins,
        station=station,
    )
    save_pkl(paths.out(f"{station}_stats_multi_meas3.pkl"), **meas3[station])
    print(f"{station}: gecombineerd met {o} (omni) en {d} (richtingen)")

## Stap 3 – Metingen van de overige stations (R-chunk `eimeas`)
Lees de gemeten hoogwaterstanden in (gecorrigeerd voor de trend, plus een heel klein toevalsgetal zoals in R).
R gebruikt deze alleen voor figuren. Eerst wordt de reeks toevalsgetallen van R voor alle stations van
`r_stations` nagespeeld, zodat elk station precies de toevalsgetallen van de R-run krijgt.

Uitvoer: `<station>_multi_meas.pkl`

In [ ]:
streams = others_random_streams(
    OTHER_SEED, r_stations, paths.input_path
)  # begint zelf met set.seed(180225)

for station in stations:
    a = read_station_wlmeas(station, paths.input_path, WL_MEASFILE)
    wlm = (
        a["wspeed"]
        + (streams[station]["eimeas"].runif(len(a["wspeed"])) - 0.5) * 0.0001
    )
    save_pkl(
        paths.out(f"{station}_multi_meas.pkl"),
        wlm=wlm,
        dtm=a["t"].to_numpy(),
        EIvaluem=WL.EIvalue,
        Nm=len(wlm),
        hwtrend=a["hwtrend"],
    )

## Figuur (controle)
De uiteindelijke herhalingstijdwaarden per richting, voor enkele herhalingstijden, per station.

In [ ]:
for station in stations:
    plot_returnlevels(
        meas3[station]["Qfreq0m"],
        bins.cats0,
        WL.freq,
        title=station,
        ylabel="hoogwaterstand [m]",
    )